## Fourier Expansion: Interval Indicator Multipliers

- This function calculates the multipliers: $\mathsf{g}_{\,j}(a, b)$ for the first `2J+1` trigonometric terms, from Fourier-series approximation of $1(a < x < b)$.
- It implements the explicit formulae described in _Lemma 3.1_ of the main-text in _Chakravorty et al. (2022)_.

### Parameters
- `a` **: float**  
  Left endpoint of the interval.
- `b` **: float**  
  Right endpoint of the interval.
- `J` **: int**  
  Number of terms in Fourier-series partial sum; the returned array is of length `2J+1`.

### Returns
- **numpy.ndarray**  
  A 1D array of length `2J+1`, storing the multipliers $\mathsf{g}_{\,j}(a, b)$ for $0\leq j\leq $`2J+1`.

### Notes
- The input specifies endpoints for the interval indicator $1(a < x < b)$.
- For input `J`, $1(a < x < b)$ is approximated by $1_{\,J}(x, a, b) = \sum_{j=0}^{2J}\mathsf{c}_{\,j}(x)\cdot \mathsf{g}_{\,j}(a, b)$, 
    - Where $\mathsf{c}_{\,0}(x) = 1$, $\mathsf{c}_{\,2j-1}(x) = \cos\big(\,(2j-1)\cdot x\,\big)$ and $\mathsf{c}_{\,2j}(x) = \sin\big(\,(2j-1)\cdot x\,\big)$ for integer $j$.
- The explicit formulae are:
    - $\mathsf{g}_0(a, b) = 1 - \frac{1}{2}\cdot\big(\,1(a > 0) + 1(b < 1)\,\big)$
    - $\mathsf{g}_{\,2j-1}(a, b) = \frac{2}{\pi\cdot(2j-1)}\cdot\Big(\,1(b < 1)\cdot\sin\big(\,(2j-1)\cdot b\,\big) - 1(a > 0)\cdot\sin\big(\,(2j-1)\cdot a\,\big)\,\Big)$ for integer $j$.
    - $\mathsf{g}_{\,2j}(a, b) = \frac{2}{\pi\cdot(2j-1)}\cdot\Big(\,1(a > 0)\cdot\cos\big(\,(2j-1)\cdot a\,\big) - 1(b < 1)\cdot\cos\big(\,(2j-1)\cdot b\,\big)\,\Big)$ for integer $j$.
- These multipliers allow efficient representation and manipulation of interval indicators for statistical estimation in distributed $k$-d tree algorithms.

### Reference
Chakravorty, A., et al. (2022), [_Scalable balanced k-d tree construction for distributed data_](https://arxiv.org/abs/2201.08288).

In [3]:
def calculate_multiplier_along_axis(a, b, J):
    """
    Calculate the first 2J+1 Fourier multipliers g_j(a, b) for the indicator 1(a < x < b)
    as in Lemma 3.1 of Chakravorty et al. (2022).
    """
    # Import all the necessary Python modules
    import numpy
    
    # Initialize numpy array with zeros 
    g = numpy.zeros(2*J+1)
    
    # Set increments for for loop
    a_inc = 2*a
    b_inc = 2*b
    
    # Set value for the constant term 
    g[0] = 1
    
    # The lower limit 'a' only influnces 'g' if a > 0
    if a > 0:
        # Adjust value for the constant term
        g[0] -= .5
        
        for j in range(1, 2 * J, 2):
            # Adjust values for the odd terms
            g[j] -= 2 / (numpy.pi * j) * numpy.sin(a)
            
            # Adjust values for the even terms
            g[j+1] += 2 / (numpy.pi * j) * numpy.cos(a)
            
            # Incremental step
            a += a_inc
    
    # The upper limit 'b' only influnces 'g' if b < 0
    if b < 1:
        # Adjust value for the constant term
        g[0] -= .5
        
        for j in range(1, 2 * J, 2):
            # Adjust values for the odd terms
            g[j] += 2 / (numpy.pi * j) * numpy.sin(b)
            
            # Adjust values for the even terms
            g[j+1] -= 2 / (numpy.pi * j) * numpy.cos(b)
            
            # Incremental step
            b += b_inc
    
    return(g)

Consider $p$-dimensional random-variable $\boldsymbol{x} := (x_{\,1},\dots,x_{\,p})'$ and $p$-dimensional interval $(\boldsymbol{a},\boldsymbol{b}) := \big(\,(a_{\,1},\dots,a_{\,p})',\,(b_{\,1},\dots,b_{\,p})'\big)$. The $k$-d tree construction involves sequintial computation of conditional medians: $\hat{m}_{\,d,\,J}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)$ for each level. The conditional median $\hat{m}_{\,d,\,J}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)$ is a solution to the equation:
$$
\frac{1}{N}\sum_{\boldsymbol{x}\in\boldsymbol{X}}\Bigg(\mathsf{1}_{\,J}(x_{\,d},a_{\,d},m)\cdot\prod_{\substack{\,1\,\leq\,t\,\leq\,p \\ \,t\,\neq\,d}}\mathsf{1}_{\,J}(x_{\,t},a_{\,t},b_{\,t})\Bigg)
=\frac{1}{N}\sum_{\boldsymbol{x}\in\boldsymbol{X}}\Bigg(\mathsf{1}_{\,J}(x_{\,d},m,b_{\,d})\cdot\prod_{\substack{\,1\,\leq\,t\,\leq\,p \\ \,t\,\neq\,d}}\mathsf{1}_{\,J}(x_{\,t},a_{\,t},b_{\,t})\Bigg)\text{ for }m\in(a_{\,d},b_{\,d})\text{ ; where }N = |\boldsymbol{X}|
$$
If we substract RHS from LHS, and expand first entires in the product summands using the expression: $1_{\,J}(x_{\,d}, a_{\,d}, b_{\,d}) = \sum_{j_d=0}^{2J}\mathsf{c}_{\,j_d}(x_{\,d})\cdot \mathsf{g}_{\,j_d}(a_{\,d}, b_{\,d})$, and interchange summation, we conclude thet $\hat{m}_{\,d,\,J}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)$ is a solution to the equivallent expression:
$$
\sum_{j_d=0}^{2J}\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)\cdot\big(\mathsf{g}_{\,j_d}(a_{\,d}, m)-\mathsf{g}_{\,j_d}(m, b_{\,d})\big)=0\text{ for }m\in(a_{\,d},b_{\,d})
\text{ ; where }\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big) := \frac{1}{N}\sum_{\boldsymbol{x}\in\boldsymbol{X}}\Bigg(\mathsf{c}_{\,j_d}(x_{\,d})\cdot\prod_{\substack{\,1\,\leq\,t\,\leq\,p \\ \,t\,\neq\,d}}\mathsf{1}_{\,J}(x_{\,t},a_{\,t},b_{\,t})\Bigg)
$$

Let $\boldsymbol{j}:=(j_1,\dots,j_p)'$ denote a general $p$-dimensional index of the index-set $\times_{t=1}^p\mathbb{N}_{\,J}$, where $\mathbb{N}_{\,J}:=\{0, 1, \cdots, 2\,J\}$, and $\boldsymbol{j}_{\text{-}d}\in\times_{t=1}^{p-1}\mathbb{N}_{\,J}$ is the index obtained by deleting the $d^{\,\text{th}}$ element from $\boldsymbol{j}$. Now, observe that
$$
\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)=\sum_{\boldsymbol{j}_{\text{-}d}\,\in\,\times_{t=1}^{p-1}\mathbb{N}_{\,J}}\bigg(\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\cdot\prod_{\substack{\,1\,\leq\,t\,\leq\,p\\ \,t\,\neq\,d}}\mathsf{g}_{\,j_t}(a_{\,t}, b_{\,t})\bigg)\text{ ; where }\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X}):=\frac{1}{N}\sum_{\boldsymbol{x}\in\boldsymbol{X}}\bigg(\prod_{\,1\,\leq\,t\,\leq\,p}\mathsf{c}_{\,j_t}(x_{\,t})\bigg)\text{ for }\boldsymbol{j}\in\times_{t=1}^p\mathbb{N}_{\,J}
$$
In our approximate $k$-d tree construction algorithm, we calculate $p$-dimensional tensor of SEP statistics: $\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}\,\in\,\times_{t=1}^p\mathbb{N}_{\,J}}$ in a single _mapReduce_ step. The function `calculate_multiplier` takes tensor $\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}\,\in\,\times_{t=1}^p\mathbb{N}_{\,J}}$ as input along with $p$-dimensional interval $(\boldsymbol{a},\boldsymbol{b})$ and the axis $d$, and returns 1D array $\big\{\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)\big\}_{\,j_d\,\in\,\mathbb{N}_{\,J}}$.

## Multipliers for $p$-dimensional Interval
- This function takes a $p$-dimensional tensor of SEP statistics $\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}\,\in\,\times_{t=1}^p\mathbb{N}_{\,J}}$ and a $p$-dimensional interval $(\boldsymbol{a},\,\boldsymbol{b})$ as input and calculates the multipliers: $\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)$, along axis $d$ for $0\leq j_{\,d}\leq 2J$.
- It applies the function `calculate_multiplier_along_axis` along every axis of $(\boldsymbol{a},\,\boldsymbol{b})$ execpt $d$, and takes an outer product of the output multipliers, follwed by an inner product with the tensor SEP statistics.

### Parameters
- `sep` **: numpy.ndarray**  
  A $p$-dimensional tensor of shape $(2\,J+1)^p$, contating SEP statistics: $p$-dimensional trigonometric moments.
- `a` **: array_like of float**  
  A sequence (tuple, list, or 1D array) of length $p$: lower limits of the interval.
- `b` **: array_like of float**  
  A sequence (tuple, list, or 1D array) of length $p$: upper limits of the interval.
- `d` **: int**  
  The axis along which multipliers are calculated.

### Returns
- **numpy.ndarray**  
  A 1D array of length $2\,J+1$, storing the multipliers $\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)$, for $0\leq j_{\,d}\leq 2\,J$.

### Notes
- The transformation `calculate_multiplier_interval` involves reshaping tensor followed by inner-product with multipliers
- First, we transform the tensor $\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}\,\in\,\times_{t=1}^p\mathbb{N}_{\,J}}$ to a $(2\,J+1)\times(2\,J+1)^{p-1}$ matrix $\{\{\bar{C}_{\,j_d\,,\,k}(\boldsymbol{X})\}\}$, with the $j_d^{\,th}$ row $\bar{C}_{\,j_d\,,\,\dots}(\boldsymbol{X}):=\text{vec}\Big(\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}_{\text{-}d}\,\in\,\times_{t=1}^{p-1}\mathbb{N}_{\,J}}\Big)$ using `numpy` functions `roleaxis` and `reshape`, here the vector $\text{vec}\big(\mathsf{T}\big)$ denotes the vectorization of the tensor $\mathsf{T}$ in the natural order.
- Next, we calculate the vectorized of $p-1$ dimensional tensor with dimension $(2\,J+1)\times\cdots\times(2\,J+1)$( $p-1$ times) $\tilde{\mathsf{I}}_{\,J,\,\text{-}d}(\boldsymbol{a},\boldsymbol{b}):=\text{vec}\Big(\Big\{\operatorname{\otimes}\limits_{\substack{\,1\,\leq\,t\,\leq\,p \\ \,t\,\neq\,d}}\big\{g_{\,j_t}(a_{\,t}, b_{\,t})\big\}_{\,j_t\,\in\,\mathbb{N}_{\,J}}\Big\}\Big)$ using `calculate_multiplier_along_axis` and `numpy` function `outer` to calculate serial outer-products( $\otimes$).
- Finaly, we take a dot product of $\{\{\bar{C}_{\,j_d\,,\,k}(\boldsymbol{X})\}\}$ and $\tilde{\mathsf{I}}_{\,J,\,\text{-}d}(\boldsymbol{a},\boldsymbol{b})$ to return $\{\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)\}_{\,j_d\in \mathbb{N}_{\,J}}$ in a vector.

In [4]:
def calculate_multipliers_conditional(sep, a, b, d):
    """
    Calculate the first 2J+1 conditional multipliers $\bar{C}_{d, j}(X, a, b)$ as seen from the calucations in the previous section.
    """
    # Import all the necessary Python modules
    import numpy
    
    # Import "epstatKDTree" module
    import epstatKDTree
         
    # Extract dimension P from sep.shape
    P = len(sep.shape)
    
    # Extract number of fourier series terms J from sep.shape
    J = int((sep.shape[0] - 1) / 2)
    
    # Transform and mold the SEP tensor into matrix
    sep_rolled = numpy.rollaxis(sep, d).reshape(2*J+1, (2*J+1)**(P-1))
    
    # Calculate the approximate indicator multiplier
    multiplier_indicator = 1
    
    # Iterate over evey dimensions of (a, b)
    for t in range(P):
        
        # Skip for t == d, take outer-product otherwise
        if t != d:
            multiplier_indicator = numpy.outer(multiplier_indicator, epstatKDTree.calculate_multiplier_along_axis(a[t], b[t], J))
    
    # Vectorize the final indicator multiplier tensor
    multiplier_indicator = multiplier_indicator.ravel()
    
    # Take dot-product of the two
    multipliers_conditional = sep_rolled.dot(multiplier_indicator)
    
    return(multipliers_conditional)

## Conditional median for $p$-dimensional Interval
- This function takes $p$-dimensional tensor of SEP statistics $\{\{\bar{\mathsf{C}}_{\,\boldsymbol{j}}(\boldsymbol{X})\}\}_{\,\boldsymbol{j}\,\in\,\times_{t=1}^p\mathbb{N}_{\,J}}$ and $(\boldsymbol{a},\,\boldsymbol{b})$ as input and calculates the approximate conditional median $\hat{m}_{\,d,\,J}(\boldsymbol{X}, \boldsymbol{a}, \boldsymbol{b})$, along axis $d$.

### Parameters
- `sep` **: numpy.ndarray**  
  A $p$-dimensional tensor of shape $(2\,J+1)^p$, contating SEP statistics: $p$-dimensional trigonometric moments.
- `a` **: array_like of float**  
  A sequence (tuple, list, or 1D array) of length $p$: lower limits of the interval.
- `b` **: array_like of float**  
  A sequence (tuple, list, or 1D array) of length $p$: upper limits of the interval.
- `d` **: int**  
  The axis along which multipliers are calculated.

### Returns
- **float**  
  Approximate conditional median $\hat{m}_{\,d,\,J}(\boldsymbol{X}, \boldsymbol{a}, \boldsymbol{b})$.

### Notes
- The fourier-series parameter $J$ is obtained from the shape of `sep`.
- It first calculates conditional multipliers $\big\{\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)\big\}_{\,j_d\,\in\,\mathbb{N}_{\,J}}$ using `calculate_multipliers_conditional` with the same input.
- Next, it solves $\sum_{j_d=0}^{2J}\bar{\mathcal{C}}_{\,d,\,j_d}\big(\boldsymbol{X},\boldsymbol{a},\boldsymbol{b}\big)\cdot\big(\mathsf{g}_{\,j_d}(a_{\,d}, m)-\mathsf{g}_{\,j_d}(m, b_{\,d})\big)=0$ for $m\in(a_{\,d},b_{\,d})$, using `scipy.optimize.brentq`, the sequences of functions: $\{\mathsf{g}_{\,j_d}(a_{\,d}, m)\}_{\,j_d\in \mathbb{N}_{\,J}}$ and $\{\mathsf{g}_{\,j_d}(m, b_{\,d})\}_{\,j_d\in \mathbb{N}_{\,J}}$ are calculated using `calculate_multiplier_along_axis`.
- The optimizer is mathematically guaranteed to exist, and is returned as function output.

In [5]:
def calculate_conditional_median(sep, a, b, d):

    # Import all the necessary Python modules
    import scipy.optimize
    
    # Import "epstatKDTree" module
    import epstatKDTree
    
    # Extract dimension p from sep.shape
    p = len(sep.shape)
    
    # Extract number of fourier series terms J from sep.shape
    J = int((sep.shape[0] - 1) / 2)

    # Calculate conditional multipliers
    if p == 1:
        # If p == 1, fetch multipliers for regular median based on fourier-series expansion
        multiplier = sep.ravel()
    else:
        # If p > 1, calculate multipliers for conditional median
        multiplier = epstatKDTree.calculate_multipliers_conditional(sep, a, b, d)

    # Define functions for brentq optimizer    
    def lambda_median(m):
        return(multiplier.dot(epstatKDTree.calculate_multiplier_along_axis(a[d], m, J) - epstatKDTree.calculate_multiplier_along_axis(m, b[d], J)))
    
    # Optimization step
    median = scipy.optimize.brentq(lambda m: lambda_median(m), a[d], b[d])
    
    return(median)